# Solve And Visualize A Thesis MILP Scenario

This notebook builds or loads one scheduling instance, solves the document-aligned MILP with Gurobi, validates the selected schedule, and visualizes baseline load, flexible load, renewable use, curtailment, grid consumption, and cluster loads.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import load_processed_inputs
from src.data.scenarios import ENERGY_SCENARIOS, build_repo_scenario
from src.data.validation import validate_clusters, validate_hourly_inputs, validate_jobs
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.plotting import plot_cluster_loads, plot_hourly_profiles, plot_schedule_gantt
from src.evaluation.results import extract_cluster_hourly_results, extract_hourly_results, extract_schedule
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model


def require_file(path: Path, label: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


## Configuration

This notebook is ready to use the copied thesis instance files under `data/instances`. Pick the exact CSV in `JOB_INSTANCE_CSV`, then pick the energy scenario whose OMIE price file and month-specific solar profile should be paired with it.

Set `BUILD_FROM_SOURCE = False` only if you want to load a fully preassembled processed folder containing `jobs.csv`, `hourly_inputs.csv`, `clusters.csv`, and `config.json`.


In [ ]:
BUILD_FROM_SOURCE = True
DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Main thesis instance selector. Change only this filename to switch job sets.
INSTANCE_DIR = PROJECT_ROOT / "data" / "instances"
JOB_INSTANCE_CSV = INSTANCE_DIR / "jobs_tense.csv"  # jobs_light.csv, jobs_tense.csv, jobs_limit.csv

# Energy scenario selector. This chooses the OMIE day and matching monthly solar profile.
SCENARIO_NAME = "base"  # one of: clear_sky, overcast, base
SOLAR_PROFILE_CSV = PROJECT_ROOT / "data" / "solar_profile" / "monthly_solar_profiles.csv"

BASELINE_LOAD_MW = 0.010      # fixed inference / Zone A baseline from the document
RENEWABLE_PRICE = 40.0        # fixed PPA/LCOE assumption, EUR/MWh
PEAK_PRICE = 1000.0           # demand-charge price applied above contracted power, EUR/MW
CONTRACTED_POWER = 0.222      # soft contracted-power threshold for excess peak charge, MW
PRICE_COLUMN = "last"        # OMIE MARGINALPDBC price column used as grid price
USE_GPU_CONSTRAINTS = True    # set False for the simplest MVP without GPU capacity limits

TIME_LIMIT_SECONDS = 60
MIP_GAP = None

available_instances = sorted(path.name for path in INSTANCE_DIR.glob("*.csv"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Mode: {'repo-local source scenario' if BUILD_FROM_SOURCE else 'processed folder'}")
print(f"Available job instances: {available_instances}")
print(f"Selected job instance: {JOB_INSTANCE_CSV}")
print(f"Available energy scenarios: {sorted(ENERGY_SCENARIOS)}")
print(f"Selected energy scenario: {SCENARIO_NAME}")
print(f"GPU constraints enabled: {USE_GPU_CONSTRAINTS}")


## Load And Validate Inputs


In [ ]:
if BUILD_FROM_SOURCE:
    require_file(JOB_INSTANCE_CSV, "job instance CSV")
    require_file(SOLAR_PROFILE_CSV, "monthly solar profile CSV")
    if SCENARIO_NAME not in ENERGY_SCENARIOS:
        raise ValueError(f"Unknown SCENARIO_NAME {SCENARIO_NAME!r}; expected one of {sorted(ENERGY_SCENARIOS)}")
    price_file = PROJECT_ROOT / ENERGY_SCENARIOS[SCENARIO_NAME]["price_file"]
    require_file(price_file, "OMIE price file")

    jobs_df, hourly_df, clusters_df, config = build_repo_scenario(
        project_root=PROJECT_ROOT,
        jobs_csv=JOB_INSTANCE_CSV,
        energy_scenario=SCENARIO_NAME,
        solar_profile_csv=SOLAR_PROFILE_CSV,
        baseline_load_mw=BASELINE_LOAD_MW,
        renewable_price=RENEWABLE_PRICE,
        peak_price=PEAK_PRICE,
        contracted_power=CONTRACTED_POWER,
        price_column=PRICE_COLUMN,
    )
else:
    jobs_df, hourly_df, clusters_df, config = load_processed_inputs(DATA_DIR)

validate_jobs(jobs_df)
validate_hourly_inputs(hourly_df)
validate_clusters(clusters_df)

baseline_load = hourly_df["baseline_load"] if "baseline_load" in hourly_df else pd.Series([0.0] * len(hourly_df))
print(f"Jobs: {len(jobs_df)}")
print(f"Hours: {len(hourly_df)}")
print(f"Flexible job energy (MWh): {(jobs_df['power'] * jobs_df['duration']).sum():.3f}")
print(f"Renewable available (MWh): {hourly_df['renewable_available'].sum():.3f}")
print(f"Baseline energy (MWh): {baseline_load.sum():.3f}")
print(f"Grid price range: {hourly_df['grid_price'].min():.2f} to {hourly_df['grid_price'].max():.2f}")

display(jobs_df.groupby("category").size().rename("count").reset_index())
display(jobs_df.head())
display(clusters_df)
display(hourly_df)
config


## Build And Solve


In [ ]:
model, variables = build_milp_model(
    jobs_df,
    hourly_df,
    clusters_df,
    config,
    enforce_gpu_constraints=USE_GPU_CONSTRAINTS,
)
model.Params.OutputFlag = 1

solve_model(model, time_limit=TIME_LIMIT_SECONDS, mip_gap=MIP_GAP)

print(f"Gurobi status: {model.Status}")
print(f"Solutions found: {model.SolCount}")
if model.SolCount == 0:
    raise RuntimeError("Gurobi did not find a feasible solution for this instance and parameter set.")
print(f"Objective value: {model.ObjVal:,.6f}")
print(f"Decision variables: {len(variables['x'])}")


Status note: if `TIME_LIMIT_SECONDS` is reached after Gurobi finds a feasible solution, the notebook still extracts and visualizes the incumbent schedule. Remove or increase the time limit to prove optimality.


## Extract Results


In [ ]:
schedule_df = extract_schedule(jobs_df, variables)
hourly_results = extract_hourly_results(hourly_df, variables)
cluster_hourly_results = extract_cluster_hourly_results(variables)
metrics = compute_summary_metrics(hourly_results, config)

display(schedule_df)
display(hourly_results)
display(cluster_hourly_results.head())


## Validation Checks


In [ ]:
schedule_with_checks = schedule_df.copy()
schedule_with_checks["compatible"] = schedule_with_checks.apply(
    lambda row: variables["compatibility"].get((row["job_id"], row["assigned_cluster"]), 0) == 1,
    axis=1,
)

capacity_violations = cluster_hourly_results[
    cluster_hourly_results["cluster_load"] > cluster_hourly_results["capacity"] + 1e-8
]
if "gpu_capacity" in cluster_hourly_results.columns:
    gpu_capacity_violations = cluster_hourly_results[
        cluster_hourly_results["cluster_gpu_load"] > cluster_hourly_results["gpu_capacity"] + 1e-8
    ]
else:
    gpu_capacity_violations = pd.DataFrame()

energy_balance_error = (
    hourly_results["renewable_consumption"]
    + hourly_results["grid_consumption"]
    - hourly_results["total_load"]
).abs().max()
print(f"All jobs assigned once: {len(schedule_df) == len(jobs_df) and schedule_df['job_id'].nunique() == len(jobs_df)}")
print(f"All assignments compatible: {schedule_with_checks['compatible'].all()}")
print(f"Cluster capacity violations: {len(capacity_violations)}")
print(f"GPU capacity violations: {len(gpu_capacity_violations)}")
print(f"Max energy balance error: {energy_balance_error:.3e}")
display(schedule_with_checks.loc[~schedule_with_checks["compatible"]])
display(capacity_violations)
display(gpu_capacity_violations)


## Metrics


In [ ]:
for key, value in metrics.items():
    print(f"{key}: {value:,.6f}")

print(f"Peak above contracted power: {metrics['peak_over_contracted']:.6f} MW")
display(schedule_df.groupby(["category", "assigned_cluster"]).size().rename("jobs").reset_index())


## Visualize Solution


In [ ]:
fig = plot_hourly_profiles(hourly_results)


In [ ]:
fig = plot_cluster_loads(cluster_hourly_results)


In [ ]:
fig = plot_schedule_gantt(schedule_df)


## Optional Export


In [ ]:
if BUILD_FROM_SOURCE:
    output_name = f"{JOB_INSTANCE_CSV.stem}_{SCENARIO_NAME}"
else:
    output_name = "processed_instance"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs" / output_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

schedule_df.to_csv(OUTPUT_DIR / "schedule.csv", index=False)
hourly_results.to_csv(OUTPUT_DIR / "hourly_results.csv", index=False)
cluster_hourly_results.to_csv(OUTPUT_DIR / "cluster_hourly_results.csv", index=False)
pd.Series(metrics).to_csv(OUTPUT_DIR / "metrics.csv", header=["value"])

print(f"Saved outputs to {OUTPUT_DIR}")
